# Task 3 of 3 (`Verify_Output`)
### Lakeflow Jobs Orchestration Lab · CDC + Medallion Architecture
**Databricks Free Edition (serverless)**

This is the **third and final task** of the Job. It **explores and validates** the processing results by displaying the contents of the Silver and Gold tables.

**Depends on:** `Process_Medallion` (Task 2).

In [0]:
# Shared configuration (identical across the 3 Job tasks)
catalog = "workspace"
schema  = "medallion_dbsql"
volume  = "raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing   = f"{base_path}/landing"     # CDC feed events are landed here

# Ensure the required structure exists (idempotent)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"USE {catalog}.{schema}")

print("catalog/schema :", f"{catalog}.{schema}")
print("landing        :", landing)

catalog/schema : workspace.medallion_dbsql
landing        : /Volumes/workspace/medallion_dbsql/raw_data/landing


## Silver (current order status)

In [0]:
%sql
SELECT order_id, customer, country, category ,product, amount, status, order_date
FROM   silver_orders
ORDER  BY order_id

order_id,customer,country,category,product,amount,status,order_date
1,Ana,Colombia,Footwear,Sneakers,90.0,pending,2026-07-01
2,Beto,Mexico,Clothing,T-Shirt,50.0,pending,2026-07-01
3,Carla,Argentina,Accessories,Cap,15.0,pending,2026-07-02
4,Diego,Chile,Clothing,Jeans,65.0,pending,2026-07-02
5,Elena,Peru,Outerwear,Jacket,120.0,pending,2026-07-03
6,Fabio,Ecuador,Bags,Backpack,55.0,pending,2026-07-03
7,Gabriela,Spain,Accessories,Watch,210.0,pending,2026-07-04
8,Hugo,United States,Accessories,Sunglasses,80.0,pending,2026-07-04
9,Isabel,Brazil,Clothing,Sweater,90.0,pending,2026-07-05
10,Javier,Uruguay,Footwear,Boots,140.0,pending,2026-07-05


## Silver (Current State of the Orders)

In [0]:
%sql
SELECT *
FROM gold_sales_region
ORDER BY country;

country,num_orders,units,total_amount,average_ticket
Argentina,1,1,15.0,15.0
Australia,1,1,180.0,180.0
Brazil,1,2,90.0,90.0
Canada,1,1,95.0,95.0
Chile,1,1,65.0,65.0
Colombia,1,1,90.0,90.0
Costa Rica,1,3,60.0,60.0
Ecuador,1,1,55.0,55.0
France,1,1,45.0,45.0
Germany,1,2,36.0,36.0


In [0]:
%sql
SELECT *
FROM gold_sales_category_day
ORDER BY order_date, category;

order_date,category,country,num_orders,units,total_amount
2026-07-01,Clothing,Mexico,1,2,50.0
2026-07-01,Footwear,Colombia,1,1,90.0
2026-07-02,Accessories,Argentina,1,1,15.0
2026-07-02,Clothing,Chile,1,1,65.0
2026-07-03,Bags,Ecuador,1,1,55.0
2026-07-03,Outerwear,Peru,1,1,120.0
2026-07-04,Accessories,Spain,1,1,210.0
2026-07-04,Accessories,United States,1,1,80.0
2026-07-05,Clothing,Brazil,1,2,90.0
2026-07-05,Footwear,Uruguay,1,1,140.0


## Simple Quality Check

A verification task typically includes **assertions**. If something is not as expected, the task **fails**, and the Job notifies you. Here, we verify that the Gold table is not empty.

In [0]:
# Uncomment to simulate a failure and practice "Repair run"
# spark.sql("SELECT * FROM table_that_does_not_exist_usa").show()

### How to Repair It
1. Run the Job with the line above **uncommented** → **Task 3** fails.
2. In **Jobs & Pipelines → Runs**, open the failed run and review the error.
3. **Comment the line again** (fixing the "bug").
4. In the failed run, click **Repair run** → only `Verify_Output` is re-executed.
5. The run completes successfully: you recovered the workflow without rerunning the entire pipeline.